<a href="https://colab.research.google.com/github/AmyMugeni/cognilens/blob/main/Cognilens_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Path to your project in Drive
project_path = '/content/drive/MyDrive/Cognilens_MLtraining'
os.makedirs(project_path, exist_ok=True)
os.chdir(project_path)

print(f"Working directory set to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory set to: /content/drive/MyDrive/Cognilens_MLtraining


## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier


## 2. Data Loading and Initial Cleaning

In [ ]:
# 1. Load Primary Dataset from the current working directory (which is your Drive folder)
df = pd.read_csv("SocialMediaUse_export (4).csv")

cols_to_drop = [
    'notification_count', 'triggered_by_notification',
    'scroll_count', 'scroll_speed_estimate', 'avg_scroll_delta_px',
    'micro_scrolls', 'normal_scrolls', 'macro_scrolls',
    'scroll_behavior', 'detected_format'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
df['minutes_since_last_session'] = df['minutes_since_last_session'].fillna(60.0)

print("Initial data loaded and cleaned.")
display(df.head())

Initial data loaded and cleaned.


,user_id,session_start,session_end,date,day_of_week,session_hour,time_of_day,app_label,duration_minutes,tap_count,minutes_since_last_session,day_asi_score,day_intensity_score,day_fragmentation_score,day_compulsion_score,day_disruption_score
0,46b5fe4a-2c09-4006-bd5b-08bf12ece5e4,1790281329502,1790281593063,2026-09-24,Thursday,23,night,Instagram,4.39,0,28.47,48,45,68,32,49
1,46b5fe4a-2c09-4006-bd5b-08bf12ece5e4,1790279631154,1790279723216,2026-09-24,Thursday,22,night,YouTube,1.53,0,33.34,48,45,68,32,49
2,46b5fe4a-2c09-4006-bd5b-08bf12ece5e4,1790279484659,1790279621543,2026-09-24,Thursday,22,night,Instagram,2.28,0,7.95,48,45,68,32,49
3,46b5fe4a-2c09-4006-bd5b-08bf12ece5e4,1790278804601,1790279007621,2026-09-24,Thursday,22,night,Instagram,3.38,2,2.64,48,45,68,32,49
4,46b5fe4a-2c09-4006-bd5b-08bf12ece5e4,1790278538451,1790278646464,2026-09-24,Thursday,22,night,Instagram,1.80,0,0.00,48,45,68,32,49


## 3. Feature Engineering

In [ ]:
# 2. Map ASI (0-100) to BSMAS (6-30)
df['bsmas_score'] = 30.0 - (df['day_asi_score'] / 100.0 * 24.0)

# Synthesize schedule conflict
np.random.seed(42)
df['is_schedule_conflict'] = df['session_hour'].apply(
    lambda h: np.random.choice([0, 1], p=[0.2, 0.8]) if (h >= 23 or h < 8) else np.random.choice([0, 1], p=[0.85, 0.15])
)

# 3. FEATURE ENGINEERING (Key to Boosting Accuracy)
df['reopen_urgency'] = df['duration_minutes'] / (df['minutes_since_last_session'] + 1.0)

# - Risk Interaction: BSMAS Score weighted by Session Duration
df['compulsion_duration_index'] = (df['bsmas_score'] / 30.0) * df['duration_minutes']

print("Feature engineering complete.")
display(df[['bsmas_score', 'is_schedule_conflict', 'reopen_urgency', 'compulsion_duration_index']].head())

Feature engineering complete.


,bsmas_score,is_schedule_conflict,reopen_urgency,compulsion_duration_index
0,18.48,1,0.148965,2.70424
1,18.48,1,0.044554,0.94248
2,18.48,0,0.254749,1.40448
3,18.48,0,0.928571,2.08208
4,18.48,0,1.800000,1.10880


## 4. Target Variable Definition

In [ ]:
# 4. Probabilistic Target Variable (Calibrated for High Precision)
def generate_high_signal_target(row):
    bsmas_factor = (row['bsmas_score'] - 6.0) / 24.0 # [0, 1]
    duration_factor = min(row['duration_minutes'] / 30.0, 1.0) # [0, 1]
    reopen_factor = 1.0 - min(row['minutes_since_last_session'] / 20.0, 1.0) # [0, 1]
    conflict_factor = float(row['is_schedule_conflict'])

    # Weighted behavioral risk
    composite_risk = (
        0.35 * bsmas_factor +
        0.30 * duration_factor +
        0.20 * reopen_factor +
        0.15 * conflict_factor
    )

    # Sharp Sigmoid Transition (Creates clear behavioral decision boundary)
    prob = 1 / (1 + np.exp(-12 * (composite_risk - 0.48)))
    return 1 if prob >= 0.5 else 0

df['should_intervene'] = df.apply(generate_high_signal_target, axis=1)

print("Target variable 'should_intervene' created.")
print(f"Target distribution:\n{df['should_intervene'].value_counts(normalize=True)}")

Target variable 'should_intervene' created.
Target distribution:
should_intervene
0    0.790837
1    0.209163
Name: proportion, dtype: float64


## 5. Prepare Features and Labels

In [ ]:
# Features array (6 features now)
feature_cols = [
    'duration_minutes',
    'minutes_since_last_session',
    'bsmas_score',
    'is_schedule_conflict',
    'reopen_urgency',
    'compulsion_duration_index'
]

X = df[feature_cols].astype(np.float32)
y = df['should_intervene'].values

# Split the dataset into training and testing sets for evaluation after tuning
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Features (X) and labels (y) prepared and split into train/test sets.")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Features (X) and labels (y) prepared and split into train/test sets.
X shape: (502, 6), y shape: (502,)


## 6. Model Training and Hyperparameter Tuning

In [12]:
# 5. Hyperparameter Tuning using GridSearchCV
print(" Tuning Gradient Boosting Hyperparameters...")
param_grid = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(gb, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train) # Use X_train, y_train for GridSearchCV

best_model = grid_search.best_estimator_
best_acc = grid_search.best_score_

print(f"\n Optimization Complete!")
print(f" Best Cross-Validation Accuracy (on training data): {best_acc * 100:.2f}%")
print(f" Best Parameters: {grid_search.best_params_}")

# Evaluate the best model on the test set
print("\n--- Model Evaluation on Test Set ---")
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])
print(f"F1-Score (Test): {f1:.4f}")
print(f"ROC-AUC (Test): {roc_auc:.4f}")

 Tuning Gradient Boosting Hyperparameters...

 Optimization Complete!
 Best Cross-Validation Accuracy (on training data): 97.01%
 Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 150, 'subsample': 0.8}

--- Model Evaluation on Test Set ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        80
           1       1.00      0.90      0.95        21

    accuracy                           0.98       101
   macro avg       0.99      0.95      0.97       101
weighted avg       0.98      0.98      0.98       101

F1-Score (Test): 0.9500
ROC-AUC (Test): 0.9994


## 7. Model Export to ONNX

In [ ]:
# 6. Export Optimized Model to ONNX
initial_type = [('float_input', FloatTensorType([None, len(feature_cols)]))] # Dynamically set input size
onnx_model = skl2onnx.convert_sklearn(
    best_model,
    initial_types=initial_type,
    target_opset=15
)

with open("cognilens_gb_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("✅ Exported optimized model to cognilens_gb_model.onnx!")

NameError: name 'FloatTensorType' is not defined